In [2]:
%pip install pymorphy3

import pandas as pd
from collections import Counter, defaultdict
from pymorphy3 import MorphAnalyzer

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
df = pd.read_csv(r"C:\Users\тема\Desktop\Реплики.csv", sep=';')
df = df.iloc[:, :2]
df.columns = ["Реплика", "Эмоция"]
df = df.dropna()

In [4]:
morph = MorphAnalyzer(lang='ru')

features_map = {
    "Род": defaultdict(Counter),
    "Время": defaultdict(Counter),
    "Лицо": defaultdict(Counter),
    "Число": defaultdict(Counter),
    "Наклонение": defaultdict(Counter),
}

for _, row in df.iterrows():
    text = str(row["Реплика"])
    emotion = str(row["Эмоция"]).strip()
    for w in text.split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            features_map["Род"][tag.gender or '-'][emotion] += 1
            features_map["Время"][tag.tense or '-'][emotion] += 1
            features_map["Лицо"][tag.person or '-'][emotion] += 1
            features_map["Число"][tag.number or '-'][emotion] += 1
            features_map["Наклонение"][tag.mood or '-'][emotion] += 1

for feature_name, data in features_map.items():
    total_all = sum(sum(c.values()) for c in data.values())
    print(f"\n{'-'*60}")
    print(f"  {feature_name}")
    print(f"{'-'*60}")
    for value, counts in sorted(data.items(), key=lambda x: -sum(x[1].values())):
        total = sum(counts.values())
        pct = total / total_all * 100 if total_all > 0 else 0
        top_str = ", ".join(f"{emo}: {cnt}" for emo, cnt in counts.most_common())
        print(f"\n  {value}: {total} ({pct:.1f}%)")
        print(f"    {top_str}")

ne_stats = Counter()
ne_total = 0
for _, row in df.iterrows():
    tokens = str(row["Реплика"]).lower().split()
    if "не" in tokens:
        emotion = str(row["Эмоция"]).strip()
        ne_stats[emotion] += 1
        ne_total += 1

print(f"\n{'-'*60}")
print(f"  Предложения с 'не'  (всего: {ne_total})")
print(f"{'-'*60}")
for emo, cnt in ne_stats.most_common():
    pct = cnt / ne_total * 100 if ne_total > 0 else 0
    print(f"  {emo}: {cnt} ({pct:.1f}%)")

for punct, label in [("?", "вопросительные"), ("!", "восклицательные"), (".", "утвердительные")]:
    punct_stats = Counter()
    punct_total = 0
    for _, row in df.iterrows():
        text = str(row["Реплика"]).strip()
        emotion = str(row["Эмоция"]).strip()
        if text.endswith(punct):
            punct_stats[emotion] += 1
            punct_total += 1
    print(f"\n{'-'*60}")
    print(f"  Предложения, оканчивающиеся на '{punct}' ({label})  (всего: {punct_total})")
    print(f"{'-'*60}")
    for emo, cnt in punct_stats.most_common():
        pct = cnt / punct_total * 100 if punct_total > 0 else 0
        print(f"  {emo}: {cnt} ({pct:.1f}%)")


------------------------------------------------------------
  Род
------------------------------------------------------------

  femn: 2796 (50.2%)
    Joy: 722, Sadness: 607, Surprise: 591, Anger: 363, Fear: 271, Neutral: 220, Неграмматично: 22

  -: 2771 (49.8%)
    Anger: 567, Sadness: 555, Joy: 549, Surprise: 517, Fear: 306, Neutral: 234, Неграмматично: 43

------------------------------------------------------------
  Время
------------------------------------------------------------

  past: 2796 (50.2%)
    Joy: 722, Sadness: 607, Surprise: 591, Anger: 363, Fear: 271, Neutral: 220, Неграмматично: 22

  -: 2760 (49.6%)
    Anger: 566, Sadness: 554, Joy: 547, Surprise: 515, Fear: 303, Neutral: 233, Неграмматично: 42

  futr: 11 (0.2%)
    Fear: 3, Surprise: 2, Joy: 2, Neutral: 1, Неграмматично: 1, Anger: 1, Sadness: 1

------------------------------------------------------------
  Лицо
------------------------------------------------------------

  -: 5556 (99.8%)
    Joy: 1269

In [5]:
df["Эмоция"] = df["Эмоция"].str.strip().str.lower()

def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"

    has_ne = "не" in str(text).lower().split()

    verb_lemma = None
    mood = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                mood = "повелительное"
            elif tag.tense == 'past':
                mood = "прошедшее"
            break
    return verb_lemma, mood, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["mood"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

def get_type(mood, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{mood}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["mood"], r["punct"], r["has_ne"]), axis=1)

BASE_TYPE = "прошедшее+утверждение"
TARGET_EMOTION = "joy"


base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == TARGET_EMOTION)].copy()
print(f"Базовых реплик (прошедшее + утверждение + joy): {len(base_df)}")

groups = df.groupby("verb_lemma")

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

transitions = defaultdict(Counter)
chains = []

for _, base_row in base_df.iterrows():
    lemma = base_row["verb_lemma"]
    if lemma is None:
        continue
    group = groups.get_group(lemma)
    
    variations = []
    for _, row in group.iterrows():
        if row["Реплика"] == base_row["Реплика"]:
            continue
        if row["type"] == BASE_TYPE:
            continue
        variations.append({
            "type": row["type"],
            "text": row["Реплика"],
            "emotion": row["Эмоция"]
        })
    
    if variations:
        chains.append({
            "base_text": base_row["Реплика"],
            "base_emotion": base_row["Эмоция"],
            "variations": variations
        })
        for var in variations:
            transitions[var["type"]][var["emotion"]] += 1

print(f"С вариациями: {len(chains)}\n")

print("=" * 70)
print("Сколько реплик сохранили Joy / сменили эмоцию")
print("=" * 70)

for op in operation_order:
    counts = transitions.get(op, Counter())
    total = sum(counts.values())
    if total == 0:
        continue
    
    stay_joy = counts.get("joy", 0)
    changed = total - stay_joy
    
    print(f"\n  Операция: {op} (всего вариаций: {total})")
    print(f"    → Joy (осталось): {stay_joy} ({stay_joy/total*100:.1f}%)")
    if changed > 0:
        print(f"    → Изменилось (всего): {changed} ({changed/total*100:.1f}%)")
        for emo, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True):
            if emo == "joy":
                continue
            print(f"        → {emo}: {cnt} ({cnt/total*100:.1f}%)")

print("\n" + "=" * 70)
print("  Примеры (первые 15)")
print("=" * 70)

for i, chain in enumerate(chains[:15]):
    print(f"\n  [База] {chain['base_text']} → {chain['base_emotion']}")
    for var in chain["variations"]:
        print(f"    [{var['type']}] {var['text']} → {var['emotion']}")

Базовых реплик (прошедшее + утверждение + joy): 91
С вариациями: 91

Сколько реплик сохранили Joy / сменили эмоцию

  Операция: прошедшее+вопрос (всего вариаций: 91)
    → Joy (осталось): 68 (74.7%)
    → Изменилось (всего): 23 (25.3%)
        → surprise: 22 (24.2%)
        → neutral: 1 (1.1%)

  Операция: прошедшее+восклицание (всего вариаций: 91)
    → Joy (осталось): 87 (95.6%)
    → Изменилось (всего): 4 (4.4%)
        → surprise: 1 (1.1%)
        → anger: 1 (1.1%)
        → sadness: 1 (1.1%)
        → неграмматично: 1 (1.1%)

  Операция: прошедшее+утверждение+не (всего вариаций: 91)
    → Joy (осталось): 2 (2.2%)
    → Изменилось (всего): 89 (97.8%)
        → sadness: 79 (86.8%)
        → neutral: 8 (8.8%)
        → anger: 2 (2.2%)

  Операция: повелительное+утверждение (всего вариаций: 87)
    → Joy (осталось): 82 (94.3%)
    → Изменилось (всего): 5 (5.7%)
        → neutral: 2 (2.3%)
        → anger: 2 (2.3%)
        → surprise: 1 (1.1%)

  Операция: повелительное+восклицание (вс

In [6]:
def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"

    has_ne = "не" in str(text).lower().split()

    verb_lemma = None
    mood = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                mood = "повелительное"
            elif tag.tense == 'past':
                mood = "прошедшее"
            break
    return verb_lemma, mood, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["mood"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

def get_type(mood, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{mood}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["mood"], r["punct"], r["has_ne"]), axis=1)

BASE_TYPE = "прошедшее+утверждение"
TARGET_EMOTION = "sadness"

base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == TARGET_EMOTION)].copy()
print(f"Базовых реплик (прошедшее + утверждение + sadness): {len(base_df)}")

groups = df.groupby("verb_lemma")

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

transitions = defaultdict(Counter)
chains = []

for _, base_row in base_df.iterrows():
    lemma = base_row["verb_lemma"]
    if lemma is None:
        continue
    group = groups.get_group(lemma)
    
    variations = []
    for _, row in group.iterrows():
        if row["Реплика"] == base_row["Реплика"]:
            continue
        if row["type"] == BASE_TYPE:
            continue
        variations.append({
            "type": row["type"],
            "text": row["Реплика"],
            "emotion": row["Эмоция"]
        })
    
    if variations:
        chains.append({
            "base_text": base_row["Реплика"],
            "base_emotion": base_row["Эмоция"],
            "variations": variations
        })
        for var in variations:
            transitions[var["type"]][var["emotion"]] += 1

print(f"С вариациями: {len(chains)}\n")

print("=" * 70)
print(" Сколько реплик сохранили Sadness / сменили эмоцию")
print("=" * 70)

for op in operation_order:
    counts = transitions.get(op, Counter())
    total = sum(counts.values())
    if total == 0:
        continue
    
    stay = counts.get("sadness", 0)
    changed = total - stay
    
    print(f"\n  Операция: {op} (всего вариаций: {total})")
    print(f"    → Sadness (осталось): {stay} ({stay/total*100:.1f}%)")
    if changed > 0:
        print(f"    → Изменилось (всего): {changed} ({changed/total*100:.1f}%)")
        for emo, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True):
            if emo == "sadness":
                continue
            print(f"        → {emo}: {cnt} ({cnt/total*100:.1f}%)")

print("\n" + "=" * 70)
print("  Примеры (первые 15)")
print("=" * 70)

for i, chain in enumerate(chains[:15]):
    print(f"\n  [База] {chain['base_text']} → {chain['base_emotion']}")
    for var in chain["variations"]:
        print(f"    [{var['type']}] {var['text']} → {var['emotion']}")


Базовых реплик (прошедшее + утверждение + sadness): 147
С вариациями: 147

 Сколько реплик сохранили Sadness / сменили эмоцию

  Операция: прошедшее+вопрос (всего вариаций: 147)
    → Sadness (осталось): 105 (71.4%)
    → Изменилось (всего): 42 (28.6%)
        → surprise: 31 (21.1%)
        → anger: 8 (5.4%)
        → fear: 2 (1.4%)
        → неграмматично: 1 (0.7%)

  Операция: прошедшее+восклицание (всего вариаций: 147)
    → Sadness (осталось): 121 (82.3%)
    → Изменилось (всего): 26 (17.7%)
        → anger: 14 (9.5%)
        → joy: 8 (5.4%)
        → fear: 3 (2.0%)
        → неграмматично: 1 (0.7%)

  Операция: прошедшее+утверждение+не (всего вариаций: 147)
    → Sadness (осталось): 4 (2.7%)
    → Изменилось (всего): 143 (97.3%)
        → joy: 125 (85.0%)
        → neutral: 13 (8.8%)
        → anger: 4 (2.7%)
        → surprise: 1 (0.7%)

  Операция: повелительное+утверждение (всего вариаций: 142)
    → Sadness (осталось): 129 (90.8%)
    → Изменилось (всего): 13 (9.2%)
        → 

In [7]:
def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"

    has_ne = "не" in str(text).lower().split()

    verb_lemma = None
    mood = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                mood = "повелительное"
            elif tag.tense == 'past':
                mood = "прошедшее"
            break
    return verb_lemma, mood, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["mood"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

def get_type(mood, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{mood}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["mood"], r["punct"], r["has_ne"]), axis=1)

BASE_TYPE = "прошедшее+утверждение"
TARGET_EMOTION = "anger"

base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == TARGET_EMOTION)].copy()
print(f"Базовых реплик (прошедшее + утверждение + anger): {len(base_df)}")

groups = df.groupby("verb_lemma")

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

transitions = defaultdict(Counter)
chains = []

for _, base_row in base_df.iterrows():
    lemma = base_row["verb_lemma"]
    if lemma is None:
        continue
    group = groups.get_group(lemma)
    
    variations = []
    for _, row in group.iterrows():
        if row["Реплика"] == base_row["Реплика"]:
            continue
        if row["type"] == BASE_TYPE:
            continue
        variations.append({
            "type": row["type"],
            "text": row["Реплика"],
            "emotion": row["Эмоция"]
        })
    
    if variations:
        chains.append({
            "base_text": base_row["Реплика"],
            "base_emotion": base_row["Эмоция"],
            "variations": variations
        })
        for var in variations:
            transitions[var["type"]][var["emotion"]] += 1

print(f"С вариациями: {len(chains)}\n")

print("=" * 70)
print("  Сколько реплик сохранили Anger / сменили эмоцию")
print("=" * 70)

for op in operation_order:
    counts = transitions.get(op, Counter())
    total = sum(counts.values())
    if total == 0:
        continue
    
    stay = counts.get("anger", 0)
    changed = total - stay
    
    print(f"\n  Операция: {op} (всего вариаций: {total})")
    print(f"    → Anger (осталось): {stay} ({stay/total*100:.1f}%)")
    if changed > 0:
        print(f"    → Изменилось (всего): {changed} ({changed/total*100:.1f}%)")
        for emo, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True):
            if emo == "anger":
                continue
            print(f"        → {emo}: {cnt} ({cnt/total*100:.1f}%)")

print("\n" + "=" * 70)
print("  Примеры (первые 15)")
print("=" * 70)

for i, chain in enumerate(chains[:15]):
    print(f"\n  [База] {chain['base_text']} → {chain['base_emotion']}")
    for var in chain["variations"]:
        print(f"    [{var['type']}] {var['text']} → {var['emotion']}")

Базовых реплик (прошедшее + утверждение + anger): 82
С вариациями: 82

  Сколько реплик сохранили Anger / сменили эмоцию

  Операция: прошедшее+вопрос (всего вариаций: 82)
    → Anger (осталось): 56 (68.3%)
    → Изменилось (всего): 26 (31.7%)
        → surprise: 14 (17.1%)
        → fear: 7 (8.5%)
        → sadness: 3 (3.7%)
        → неграмматично: 1 (1.2%)
        → joy: 1 (1.2%)

  Операция: прошедшее+восклицание (всего вариаций: 82)
    → Anger (осталось): 72 (87.8%)
    → Изменилось (всего): 10 (12.2%)
        → joy: 3 (3.7%)
        → neutral: 2 (2.4%)
        → surprise: 2 (2.4%)
        → fear: 2 (2.4%)
        → sadness: 1 (1.2%)

  Операция: прошедшее+утверждение+не (всего вариаций: 82)
    → Anger (осталось): 2 (2.4%)
    → Изменилось (всего): 80 (97.6%)
        → joy: 45 (54.9%)
        → neutral: 31 (37.8%)
        → sadness: 3 (3.7%)
        → неграмматично: 1 (1.2%)

  Операция: повелительное+утверждение (всего вариаций: 81)
    → Anger (осталось): 72 (88.9%)
    → Изме

In [8]:
def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"

    has_ne = "не" in str(text).lower().split()

    verb_lemma = None
    mood = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                mood = "повелительное"
            elif tag.tense == 'past':
                mood = "прошедшее"
            break
    return verb_lemma, mood, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["mood"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

def get_type(mood, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{mood}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["mood"], r["punct"], r["has_ne"]), axis=1)

BASE_TYPE = "прошедшее+утверждение"
TARGET_EMOTION = "fear"

base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == TARGET_EMOTION)].copy()
print(f"Базовых реплик (прошедшее + утверждение + fear): {len(base_df)}")

groups = df.groupby("verb_lemma")

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

transitions = defaultdict(Counter)
chains = []

for _, base_row in base_df.iterrows():
    lemma = base_row["verb_lemma"]
    if lemma is None:
        continue
    group = groups.get_group(lemma)
    
    variations = []
    for _, row in group.iterrows():
        if row["Реплика"] == base_row["Реплика"]:
            continue
        if row["type"] == BASE_TYPE:
            continue
        variations.append({
            "type": row["type"],
            "text": row["Реплика"],
            "emotion": row["Эмоция"]
        })
    
    if variations:
        chains.append({
            "base_text": base_row["Реплика"],
            "base_emotion": base_row["Эмоция"],
            "variations": variations
        })
        for var in variations:
            transitions[var["type"]][var["emotion"]] += 1

print(f"С вариациями: {len(chains)}\n")

print("=" * 70)
print("  Сколько реплик сохранили Fear / сменили эмоцию")
print("=" * 70)

for op in operation_order:
    counts = transitions.get(op, Counter())
    total = sum(counts.values())
    if total == 0:
        continue
    
    stay = counts.get("fear", 0)
    changed = total - stay
    
    print(f"\n  Операция: {op} (всего вариаций: {total})")
    print(f"    → Fear (осталось): {stay} ({stay/total*100:.1f}%)")
    if changed > 0:
        print(f"    → Изменилось (всего): {changed} ({changed/total*100:.1f}%)")
        for emo, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True):
            if emo == "fear":
                continue
            print(f"        → {emo}: {cnt} ({cnt/total*100:.1f}%)")

print("\n" + "=" * 70)
print("  Примеры (первые 15)")
print("=" * 70)

for i, chain in enumerate(chains[:15]):
    print(f"\n  [База] {chain['base_text']} → {chain['base_emotion']}")
    for var in chain["variations"]:
        print(f"    [{var['type']}] {var['text']} → {var['emotion']}")

Базовых реплик (прошедшее + утверждение + fear): 83
С вариациями: 83

  Сколько реплик сохранили Fear / сменили эмоцию

  Операция: прошедшее+вопрос (всего вариаций: 83)
    → Fear (осталось): 60 (72.3%)
    → Изменилось (всего): 23 (27.7%)
        → surprise: 21 (25.3%)
        → sadness: 1 (1.2%)
        → anger: 1 (1.2%)

  Операция: прошедшее+восклицание (всего вариаций: 83)
    → Fear (осталось): 75 (90.4%)
    → Изменилось (всего): 8 (9.6%)
        → joy: 3 (3.6%)
        → anger: 2 (2.4%)
        → sadness: 1 (1.2%)
        → неграмматично: 1 (1.2%)
        → surprise: 1 (1.2%)

  Операция: прошедшее+утверждение+не (всего вариаций: 83)
    → Fear (осталось): 0 (0.0%)
    → Изменилось (всего): 83 (100.0%)
        → neutral: 43 (51.8%)
        → joy: 37 (44.6%)
        → sadness: 2 (2.4%)
        → неграмматично: 1 (1.2%)

  Операция: повелительное+утверждение (всего вариаций: 81)
    → Fear (осталось): 73 (90.1%)
    → Изменилось (всего): 8 (9.9%)
        → sadness: 3 (3.7%)
    

In [9]:
def analyze_row(text):
    text = str(text).strip()
    if text.endswith("?"):
        punct = "вопрос"
    elif text.endswith("!"):
        punct = "восклицание"
    elif text.endswith("."):
        punct = "утверждение"
    else:
        punct = "прочее"

    has_ne = "не" in str(text).lower().split()

    verb_lemma = None
    mood = "прочее"
    for w in str(text).split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            verb_lemma = p.normal_form
            if tag.mood == 'impr':
                mood = "повелительное"
            elif tag.tense == 'past':
                mood = "прошедшее"
            break
    return verb_lemma, mood, punct, has_ne

info = df["Реплика"].apply(analyze_row)
df["verb_lemma"] = [x[0] for x in info]
df["mood"] = [x[1] for x in info]
df["punct"] = [x[2] for x in info]
df["has_ne"] = [x[3] for x in info]

def get_type(mood, punct, has_ne):
    ne_str = "+не" if has_ne else ""
    return f"{mood}+{punct}{ne_str}"

df["type"] = df.apply(lambda r: get_type(r["mood"], r["punct"], r["has_ne"]), axis=1)

BASE_TYPE = "прошедшее+утверждение"
TARGET_EMOTION = "surprise"

base_df = df[(df["type"] == BASE_TYPE) & (df["Эмоция"] == TARGET_EMOTION)].copy()
print(f"Базовых реплик (прошедшее + утверждение + surprise): {len(base_df)}")

groups = df.groupby("verb_lemma")

operation_order = [
    "прошедшее+вопрос",
    "прошедшее+восклицание",
    "прошедшее+утверждение+не",
    "повелительное+утверждение",
    "повелительное+восклицание",
    "повелительное+вопрос",
    "повелительное+утверждение+не",
    "повелительное+восклицание+не",
    "повелительное+вопрос+не",
]

transitions = defaultdict(Counter)
chains = []

for _, base_row in base_df.iterrows():
    lemma = base_row["verb_lemma"]
    if lemma is None:
        continue
    group = groups.get_group(lemma)
    
    variations = []
    for _, row in group.iterrows():
        if row["Реплика"] == base_row["Реплика"]:
            continue
        if row["type"] == BASE_TYPE:
            continue
        variations.append({
            "type": row["type"],
            "text": row["Реплика"],
            "emotion": row["Эмоция"]
        })
    
    if variations:
        chains.append({
            "base_text": base_row["Реплика"],
            "base_emotion": base_row["Эмоция"],
            "variations": variations
        })
        for var in variations:
            transitions[var["type"]][var["emotion"]] += 1

print(f"С вариациями: {len(chains)}\n")

print("=" * 70)
print("  Сколько реплик сохранили Surprise / сменили эмоцию")
print("=" * 70)

for op in operation_order:
    counts = transitions.get(op, Counter())
    total = sum(counts.values())
    if total == 0:
        continue
    
    stay = counts.get("surprise", 0)
    changed = total - stay
    
    print(f"\n  Операция: {op} (всего вариаций: {total})")
    print(f"    → Surprise (осталось): {stay} ({stay/total*100:.1f}%)")
    if changed > 0:
        print(f"    → Изменилось (всего): {changed} ({changed/total*100:.1f}%)")
        for emo, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True):
            if emo == "surprise":
                continue
            print(f"        → {emo}: {cnt} ({cnt/total*100:.1f}%)")

print("\n" + "=" * 70)
print("  Примеры (первые 15)")
print("=" * 70)

for i, chain in enumerate(chains[:15]):
    print(f"\n  [База] {chain['base_text']} → {chain['base_emotion']}")
    for var in chain["variations"]:
        print(f"    [{var['type']}] {var['text']} → {var['emotion']}")


Базовых реплик (прошедшее + утверждение + surprise): 23
С вариациями: 23

  Сколько реплик сохранили Surprise / сменили эмоцию

  Операция: прошедшее+вопрос (всего вариаций: 23)
    → Surprise (осталось): 18 (78.3%)
    → Изменилось (всего): 5 (21.7%)
        → неграмматично: 2 (8.7%)
        → fear: 2 (8.7%)
        → anger: 1 (4.3%)

  Операция: прошедшее+восклицание (всего вариаций: 23)
    → Surprise (осталось): 20 (87.0%)
    → Изменилось (всего): 3 (13.0%)
        → fear: 3 (13.0%)

  Операция: прошедшее+утверждение+не (всего вариаций: 23)
    → Surprise (осталось): 0 (0.0%)
    → Изменилось (всего): 23 (100.0%)
        → neutral: 13 (56.5%)
        → sadness: 8 (34.8%)
        → joy: 2 (8.7%)

  Операция: повелительное+утверждение (всего вариаций: 22)
    → Surprise (осталось): 18 (81.8%)
    → Изменилось (всего): 4 (18.2%)
        → fear: 2 (9.1%)
        → joy: 1 (4.5%)
        → sadness: 1 (4.5%)

  Операция: повелительное+восклицание (всего вариаций: 22)
    → Surprise (оста